# PIB desde World Bank/WDI

Este notebook muestra el flujo basico de `economista`: descargar datos de World Bank/WDI, convertir el resultado a pandas y trabajar con unidades legibles.

## 1. Importar la API de datos

`economista.data` concentra la interfaz publica para buscar y descargar datos economicos.

In [ ]:
from economista import data

## 2. Descargar PIB corriente

El indicador `NY.GDP.MKTP.CD` corresponde a PIB en dolares corrientes. El resultado es un `EconDataset`, no un `DataFrame` plano: trae datos, metadata, esquema e historial minimo de consulta.

In [ ]:
data_gdp = data.fetch(
    source="world_bank",
    dataset="wdi",
    indicator="NY.GDP.MKTP.CD",
    geo=["MEX", "COL", "ARG", "GBR"],
    start=2000,
    end=2024,
)

data_gdp.metadata.indicator_name

## 3. Convertir a pandas

Para explorar o transformar los datos, usa `to_pandas()`. La columna `value` conserva el valor original de World Bank.

In [ ]:
df_gdp = data_gdp.to_pandas()

df_gdp[["geo", "geo_name", "time", "value", "indicator_name"]].head()

## 4. Mostrar el PIB en billions

World Bank entrega el PIB en dolares. Para lectura rapida, lo convertimos a billions de USD: `value / 1_000_000_000`.

In [ ]:
df_gdp_display = (
    df_gdp.assign(value_usd_billions=df_gdp["value"] / 1_000_000_000)
    .drop(columns="value")
    .sort_values(["geo", "time"])
)

df_gdp_display.head().round({"value_usd_billions": 1})

## 5. Comparar paises en formato ancho

El formato canonico de `economista` es largo. Para comparar paises por anio, podemos pivotear temporalmente.

In [ ]:
gdp_usd_billions = (
    df_gdp.assign(value_usd_billions=df_gdp["value"] / 1_000_000_000)
    .pivot(index="time", columns="geo", values="value_usd_billions")
    .sort_index()
)

gdp_usd_billions.tail().round(1)

## 6. Revisar metadata

La metadata guarda fuente, dataset, indicador, cobertura geografica, cobertura temporal, fecha de descarga y parametros de consulta.

In [ ]:
data_gdp.metadata.to_dict()

## 7. Buscar indicadores relacionados

`data.search` ayuda a encontrar codigos de indicadores dentro de World Bank/WDI.

In [ ]:
data.search(
    source="world_bank",
    query="GDP current US dollars",
    limit=10,
)